# Chapter 17 — LLM 기반 로그 해석 · 프롬프트 엔지니어링 (실습)

> **주의**: 실제 Claude API 호출에는 API key 가 필요하다 (https://console.anthropic.com).
> 본 실습은 키가 있으면 실제 호출, 없으면 모의 응답으로 진행.

## 실습 목표
1. Claude API 첫 호출
2. 시스템 프롬프트 + JSON 출력
3. 50건 일괄 분류
4. ML 1차 + LLM 2차 하이브리드
5. Prompt caching 비용 비교

> **참고 — Part IV 모델 티어 선택**
>
> 태스크에 맞춰 모델을 나눠 쓴다. 분류처럼 대량·정형 작업은 저비용 **Haiku**,
> RAG 해석·Agent 추론처럼 판단이 무거운 작업은 **Sonnet 5·Opus 4.8**.
> 단 현행 SDK 와 최신 모델은 `temperature`·`top_p`·`top_k` 를 받지 않는다. 호출에서 생략하며,
> 같은 입력에 같은 출력을 보장하는 설정은 없다고 보고 설계한다.

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q anthropic
# !pip install -q pandas
# !pip install -q numpy

In [ ]:
import os, json, re, time
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

# --- API 키 로드: 환경변수 → 루트 .env → Colab Secrets → getpass (없으면 mock 자동) ---
LLM_MODE = os.environ.get('BOOK_LLM_MODE', 'auto')
if LLM_MODE != 'mock':
    if not os.environ.get('ANTHROPIC_API_KEY'):
        from pathlib import Path as _P
        for _d in [_P.cwd(), *_P.cwd().parents]:
            _e = _d / '.env'
            if _e.exists():
                for _l in _e.read_text(encoding='utf-8').splitlines():
                    if _l.strip().startswith('ANTHROPIC_API_KEY='):
                        os.environ['ANTHROPIC_API_KEY'] = _l.split('=', 1)[1].strip().strip('"').strip("'")
                break
        if not os.environ.get('ANTHROPIC_API_KEY'):
            try:
                from google.colab import userdata
                os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
            except Exception:
                pass
        if not os.environ.get('ANTHROPIC_API_KEY'):
            try:  # 대화형(Jupyter/Colab)에서만 입력받는다. Enter 만 치면 mock 으로 진행.
                import getpass
                os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY (Enter=mock): ') or ''
            except Exception:
                pass
API_KEY = os.environ.get('ANTHROPIC_API_KEY') if os.environ.get('BOOK_LLM_MODE') != 'mock' else None  # Colab Secrets 또는 환경변수
USE_REAL_API = bool(API_KEY)

if USE_REAL_API:
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEY)
    print('Real API mode')
else:
    print('Mock mode (API key 없음)')

In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
# 그래프에서 한글이 깨지면(사각형 □) 아래 설치 명령의 주석을 1회 해제해 실행한 뒤 런타임을 재시작한다.
#   Colab / Ubuntu:  !apt-get install -y fonts-noto-cjk fonts-nanum > /dev/null 2>&1
#   pip 대안       :  !pip install koreanize-matplotlib -q   →   import koreanize_matplotlib
# Windows(맑은 고딕)·macOS(Apple SD Gothic)는 대개 한글 폰트가 기본 설치돼 있어 추가 설치가 필요 없다.
import os
import matplotlib
from matplotlib import font_manager

# OS별로 설치돼 있을 만한 한글 폰트 파일을 모두 등록 시도한다(Noto·나눔·맑은고딕·굴림·Apple 등).
for _fp in [
    # Linux
    '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
    '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf',
    # Windows
    'C:/Windows/Fonts/malgun.ttf', 'C:/Windows/Fonts/malgunbd.ttf',
    'C:/Windows/Fonts/gulim.ttc', 'C:/Windows/Fonts/batang.ttc',
    # macOS
    '/System/Library/Fonts/AppleSDGothicNeo.ttc',
    '/System/Library/Fonts/Supplemental/AppleGothic.ttf',
    '/Library/Fonts/AppleGothic.ttf',
]:
    if os.path.exists(_fp):
        try:
            font_manager.fontManager.addfont(_fp)
        except Exception:
            pass

# 등록된 폰트 중 사용 가능한 한글 폰트를 우선순위대로 선택한다.
_avail = {f.name for f in font_manager.fontManager.ttflist}
for _name in ['Malgun Gothic', '맑은 고딕', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans KR',
              'Apple SD Gothic Neo', 'AppleGothic', 'NanumBarunGothic', 'Gulim', 'Batang',
              'Noto Sans CJK JP']:
    if _name in _avail:
        matplotlib.rcParams['font.family'] = _name
        break
matplotlib.rcParams['axes.unicode_minus'] = False  # 한글 폰트에서 마이너스 기호 깨짐 방지


---
## Step 1. 시스템 프롬프트 설정

In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. 시스템 프롬프트 설정
# 만들 결과: SYSTEM_PROMPT
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
SYSTEM_PROMPT = ...
...
규칙: 확신 없으면 UNKNOWN, severity = ...
print(f'system prompt length = ...
```
</details>


---
## Step 2. 합성 로그 50건

In [ ]:
rng = np.random.default_rng(42)
logs = []
for _ in range(30):
    logs.append({'src': f'10.0.0.{rng.integers(1,50)}', 'method': 'GET',
                 'url': rng.choice(['/index.html', '/products', '/api/v1/users']),
                 'ua': 'Mozilla/5.0', 'status': 200})
for _ in range(10):
    logs.append({'src': '10.0.0.99', 'method': 'POST',
                 'url': rng.choice(["/login?id=' OR 1=1--", '/search?q=<script>alert(1)</script>',
                                     '/../etc/passwd', '/login?id=1 UNION SELECT *']),
                 'ua': rng.choice(['sqlmap/1.5', 'nikto/2.1', 'curl/7.0']),
                 'status': rng.choice([401, 403, 500])})
for _ in range(10):
    logs.append({'src': f'192.168.1.{rng.integers(100,200)}', 'method': 'GET',
                 'url': f'/login?attempt={rng.integers(1,100)}',
                 'ua': 'Mozilla/5.0', 'status': 401})
print(f'총 {len(logs)} 건')
logs[:3]

---
## Step 3. 분류 함수 — 실제 API 또는 모의 응답

In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. 분류 함수 — 실제 API 또는 모의 응답
# 만들 결과: mock_classify(), parse_classification(), unknown, match, result, category
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def mock_classify(log: ...
    ...
    url, ua = ...
    if re.search(r'union|select|or\s+1=1', url, re.I):
        return ...
                ...
    if re.search(r'<script>|javascript: ...
        return ...
                ...
    if '../' in url:
        return ...
                ...
    if re.search(r'sqlmap|nikto', ua, re.I):
        return ...
                ...
    return ...
            ...

def parse_classification(text, category_key):
    unknown = ...
               ...
    try:
        match = ...
        result = ...
        if not isinstance(result, dict):
            return ...
        category = ...
        if not isinstance(category, str) or category not in {
                ...
            return ...
        severity = ...
        if category == 'UNKNOWN':
            return ...
        if type(severity) is not int or not 1 <= severity <= 5:
            return ...
        for key in ('summary', 'reasoning'):
            if key in result and not isinstance(result[key], str):
                return ...
        for key in ('indicators', 'attack_techniques', 'recommended_actions'):
            if key in result and (not isinstance(result[key], list) or
                                  ...
                return ...
        return ...
    except (ValueError, TypeError):
        return ...

def real_classify(log: ...
    try:
        resp = ...
            model = ...
            max_tokens = ...
            system = ...
                     ...
            messages = ...
        ...
        text = ...
        return ...
    except Exception:
        return ...

classify = ...
```
</details>


In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. 분류 함수 — 실제 API 또는 모의 응답
# 만들 결과: start, results, elapsed, df
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# 50건 일괄 분류 (병렬)
start = ...
if USE_REAL_API:
    with ThreadPoolExecutor(max_workers=5) as ex:
        results = ...
else:
    results = ...
elapsed = ...

df = ...
...
```
</details>


In [ ]:
# 위협 카테고리 분포 시각화
import matplotlib.pyplot as plt
vc = df['threat_category'].value_counts()
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(vc.index.astype(str), vc.values, color='#1a5f7a')
ax.set_title(f'위협 카테고리 분포 (LLM 분류, n={len(df)}건)')
ax.set_xlabel('카테고리'); ax.set_ylabel('건수')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

---
## Step 4. ML 1차 + LLM 2차 하이브리드 시뮬레이션

In [ ]:
# ── 실습 과제 ──
# 목표: Step 4. ML 1차 + LLM 2차 하이브리드 시뮬레이션
# 만들 결과: ml_score(), rows, s, r, routing
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# ML 1차 — 단순 룰 기반 확률 mock
def ml_score(log: ...
    url, ua = ...
    if re.search(r'union|select|<script>|\.\./', url, re.I): ...
    if re.search(r'sqlmap|nikto', ua, re.I): ...
    if log.get('status') == 401: ...
    return ...

rows = ...
for log in logs:
    s = ...
    if s < 0.3:
        ...
    elif s > 0.7:
        ...
    else:
        r = ...
        ...

routing = ...
...
```
</details>


---
## 캐싱 실측 — 긴 시스템 프롬프트일 때만 효과

`cache_control` 은 시스템 프롬프트가 캐시 최소 길이(측정상 Haiku 약 4천 토큰) 이상일 때만 동작한다.
앞의 분류용 프롬프트(약 190 토큰)는 짧아 캐시가 안 걸린다. 아래는 실제 SOC 수준의 긴 룰북을 붙여
캐시를 발동시키고, 2회 호출의 토큰 사용량으로 절감을 실측한다(캐시 읽기 단가 ≈ 입력 단가의 10%).

In [ ]:
# ── 실습 과제 ──
# 목표: 캐싱 실측 — 긴 시스템 프롬프트일 때만 효과 — `cache_control` 은 시스템 프롬프트가 캐시 최소 길이(측정상 Haiku 약 4천 토큰) 이상일 때만 동작한다.
# 만들 결과: RUBRIC, LONG_SYSTEM, _client, _call(), model, system
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# 캐시 발동 최소 길이를 넘기기 위한 긴 시스템 프롬프트 = 짧은 지침 + 상세 룰북
RUBRIC = ...
    ...
    "입력 로그를 일관된 기준으로 분류한다. 확신이 낮으면 항상 보수적으로 BENIGN, severity = ...
    ...
LONG_SYSTEM = ...

if USE_REAL_API:
    from anthropic import Anthropic
    _client = ...
    def _call():
        return ...
            model = ...
            system = ...
                     ...
            messages = ...
    u1 = ...
    def g(u, k): ...
    print(f'1차 호출 — cache_creation = ...
    print(f'2차 호출 — cache_creation = ...
    reused = ...
    ...
else:
    ...
```
</details>


---
## 정리
- LLM 으로 보안 로그를 분류·설명·대응 추천까지 한 번에.
- 하이브리드 (ML 빠른 필터 + LLM 모호 케이스 분석) 가 비용·속도·설명 균형.
- Prompt caching 은 긴 시스템 프롬프트를 반복 사용할 때만 절감된다(아래 '캐싱 실측' 참고). 짧은 프롬프트엔 효과가 없다.
- 이어지는 후반부에서 프롬프트 엔지니어링(baseline·CoT·few-shot) 패턴을 실습한다.

---

## (이어서) 프롬프트 엔지니어링 심화와 XAI

> 본 절은 Claude API 기초 위에서 Role·Few-shot·CoT·구조화 출력 4 패턴과 설명 가능성(XAI)·프롬프트 A/B 를 다룬다.



## 실습 목표
1. 4 변형 (baseline / CoT / few-shot / full)
2. 30건 × 4번 호출, 정확도 비교
3. 설명 품질 점수
4. 최고 변형 선정

In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q anthropic
# !pip install -q pandas
# !pip install -q numpy
# !pip install -q matplotlib

In [ ]:
import os, json, re
import numpy as np
import pandas as pd
API_KEY = os.environ.get('ANTHROPIC_API_KEY') if os.environ.get('BOOK_LLM_MODE') != 'mock' else None
USE_REAL = bool(API_KEY)
if USE_REAL:
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEY)

In [ ]:
# 평가용 라벨된 로그 30건 (다양한 카테고리)
rng = np.random.default_rng(42)
labeled = []
for _ in range(10):
    labeled.append(({'src':'10.0.0.5','url':'/index.html','ua':'Mozilla/5.0','status':200}, 'BENIGN'))
for _ in range(7):
    labeled.append(({'src':'10.0.0.99','url':"/login?id=' OR 1=1--",'ua':'sqlmap/1.5','status':401}, 'SQLi'))
for _ in range(7):
    labeled.append(({'src':'10.0.0.99','url':'/?q=<script>alert(1)</script>','ua':'curl','status':200}, 'XSS'))
for _ in range(6):
    labeled.append(({'src':'192.168.1.50','url':'/login','ua':'Mozilla/5.0','status':401}, 'Brute'))
rng.shuffle(labeled)
print(f'총 {len(labeled)} 건')

---
## Step 1. 4가지 프롬프트 변형

In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. 4가지 프롬프트 변형
# 만들 결과: BASELINE, COT, EXAMPLES, FEWSHOT, FULL, PROMPTS
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
BASELINE = ...
...

COT = ...

EXAMPLES = ...
...
FEWSHOT = ...
FULL = ...

PROMPTS = ...
```
</details>


---
## Step 2. 분류 함수 (mock + real)

In [ ]:
# ── 실습 과제 ──
# 목표: Step 2. 분류 함수 (mock + real)
# 만들 결과: mock_classify(), has_examples, has_cot, cat, real_classify(), resp
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def mock_classify(log, system):
    url, ua = ...
    has_examples = ...
    has_cot = ...
    # 더 정교한 프롬프트일수록 mock 정확도 ↑
    if re.search(r'union|select|or\s+1=1', url, re.I):
        cat = ...
    elif re.search(r'<script>|javascript: ...
        cat = ...
    elif log.get('status') == 401 and 'login' in url:
        cat = ...
    else:
        cat = ...
    return ...
            ...

def real_classify(log, system):
    try:
        resp = ...
            max_tokens = ...
            messages = ...
        text = ...
        return ...
    except Exception:
        return ...

classify = ...
```
</details>


---
## Step 3. A/B 평가

In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. A/B 평가
# 만들 결과: results, correct, r, results[name], color
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
results = ...
for name, sys in PROMPTS.items():
    correct = ...
    for log, truth in labeled:
        r = ...
        if r.get('category') == truth:
            correct += ...
    results[name] = ...
    print(f'{name:10s} accuracy = ...

import matplotlib.pyplot as plt
fig, ax = ...
...
        color = ...
for i, (k, v) in enumerate(results.items()):
    ...
```
</details>


---
## Step 4. 설명 품질 평가

In [ ]:
# ── 실습 과제 ──
# 목표: Step 4. 설명 품질 평가
# 만들 결과: eval_explanation(), analysis, reasoning, techniques, actions, best_prompt
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def eval_explanation(log, analysis):
    analysis = ...
    reasoning = ...
    techniques = ...
    actions = ...
    return ...
        ...

best_prompt = ...
rows = ...
for log, truth in labeled[: ...
    a = ...
    ...
```
</details>


---
## 정리
- 프롬프트 변형은 같은 표본으로 비교하며, 결합 방식이 항상 우수하지는 않다.
- 설명 길이·기법 ID·대응 필드는 형식 점검이다. 설명의 사실성과 충실성은 근거와 대조한다.
- 다음 ch18에서는 검색 자료를 연결하고 답변의 인용 ID를 검증한다.
- 이전 실제 API 측정(30건)은 기준·few-shot·결합 80.00%, CoT 66.67%였다. 현재 출력의 실행 모드와 구분한다.
